In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
# Scikit-Learn Ecosystem
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the CSV file
import os
import pandas as pd
q1_path = os.path.join(path, 'Q1_data.csv')
q1_data = pd.read_csv(q1_path)

In [ ]:
# Task 2: Write your code here:
q1_data.head()

In [ ]:
# Task 3: Write your code here:
q1_data.info()

In [ ]:
# Task 4: Write your code here:
q1_data.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(q1_data['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
q1_data = q1_data.drop(['Order_ID'], axis=1)
q1_data.head()

In [ ]:
# Task 2: Write your code here:
# Analyzing missing values
missing_percentage = (q1_data.isnull().sum() / len(q1_data)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

# Drop rows where target (Delivery Time) or key features are missing - can't predict without them
print(f"Before: {q1_data.shape}")
q1_data = q1_data.dropna(subset=['Delivery_Time','Courier_Experience_yrs','Traffic_Level','Time_of_Day'])
print(f"After dropping missing Delivery time, Courier Experience yrs, Traffic Level and Time of Day: {q1_data.shape}")

In [ ]:
# Task 3: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(q1_data)

In [ ]:
# Task 4: Write your code here:
# Let's first check if we have categorical columns. We do.
categorical_cols = q1_data.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  q1_data[col] = le.fit_transform(q1_data[col])
  label_encoders[col] = le

q1_data

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = q1_data.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
q1_data[numerical_cols] = scaler.fit_transform(q1_data[numerical_cols])
q1_data.head()


In [ ]:
# Task 6: Write your code here:
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=q1_data[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(q1_data, "Delivery_Time")

# The data looks pretty okay, little heavy on the left, the right side is lighter.

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import StratifiedKFold

# Define features and target
X = q1_data.drop("Delivery_Time", axis=1).astype(float)
y = q1_data['Delivery_Time'].astype(float)

n_splits = 5 # K=5 Folds

In [ ]:
# Task 2,3,4,5: Write your code here:

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

mae_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)


    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("Model trained!")

# Evaluation
print(f"MAE (Avg Error): {mean_absolute_error(y_test, y_pred)}")

In [ ]:
feature_cols =['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',	'Vehicle_Type', 'Preparation_Time_min',	'Delivery_Time']
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: